---
## 1. Chunking Large CSVs and JSONs

### 1. Definition
Chunking processes massive flat files (CSV, JSON) in incremental batches using a memory iterator (e.g., `chunksize` parameter), allowing systems with constrained RAM to analyze datasets larger than physical memory.

###  Naive Example
Reading a 10 GB transaction log CSV file in chunks of 100,000 rows to compute aggregate daily revenues without triggering an out-of-memory crash.

###  Uses
- Processing multi-gigabyte flat files on standard laptops or worker nodes.
- Streaming incremental data transformations and filtering passes.
- Avoiding system memory overflow during exploratory data ingest.

In [1]:
import pandas as pd
import numpy as np
import io

csv_data = "id,val\n1,10\n2,20\n3,30\n4,40\n5,50"
chunk_size = 2
total_sum = 0

# Simulate chunked reading
for chunk in pd.read_csv(io.StringIO(csv_data), chunksize=chunk_size):
    total_sum += chunk['val'].sum()
print(f"Computed Total via Chunking: {total_sum}")

Computed Total via Chunking: 150


---
## 2. Columnar Storage with Parquet and Feather

### 1. Definition
Columnar storage formats (Apache Parquet, Apache Arrow Feather) store data organized by columns rather than rows, enabling lightning-fast read speeds, data compression, and native type preservation.

###  Naive Example
Saving a 5-million-row DataFrame as a `.parquet` file instead of `.csv` to reduce file size by 75% and load back into Pandas 10 times faster.

###  Uses
- High-performance analytical querying and data lake storage.
- Preserving exact Pandas data types (integers, categories, timestamps) across sessions.
- Minimizing disk I/O bottlenecks in data engineering pipelines.


In [2]:
df_io = pd.DataFrame({
    'category': ['A', 'B', 'C'] * 100,
    'value': np.random.randn(300)
})

# Parquet read/write simulation (using pyarrow/fastparquet if available, fallback buffer test)
buffer = io.BytesIO()
df_io.to_parquet(buffer, compression='snappy')
buffer.seek(0)
df_recovered = pd.read_parquet(buffer)
print("Recovered Parquet Shape:", df_recovered.shape)


Recovered Parquet Shape: (300, 2)


---
## 3. Reading and Writing Compressed Data Files

### 1. Definition
Compressed data I/O integrates on-the-fly compression codecs (gzip, zstd, bz2, zip) directly into file read and write operations, balancing CPU decompression overhead against disk storage space savings.

In [3]:
df_comp = pd.DataFrame({'text': ['sample data record'] * 1000})

# Write and read gzipped CSV
gz_buffer = io.BytesIO()
df_comp.to_csv(gz_buffer, compression='gzip', index=False)
gz_buffer.seek(0)
df_read_gz = pd.read_csv(gz_buffer, compression='gzip')
print("Compressed Read Success. Rows:", len(df_read_gz))


Compressed Read Success. Rows: 1000


##  Incremental Data Processing Best Practices

###  Definition
Incremental data processing (or micro-batching) processes incoming data streams or partitioned subsets sequentially rather than reloading and re-computing entire historical datasets from scratch.

###  Naive Example
Appending only today's new sales transactions to a partitioned master parquet dataset rather than rewriting the entire multi-year database.

###  Uses
- Building scalable ETL pipelines and real-time dashboards.
- Minimizing compute costs and execution times in production data workflows.
- Handling append-only log files efficiently.

In [4]:
historical_state = pd.DataFrame({'day': ['Mon', 'Tue'], 'sales': [100, 150]})
delta_batch = pd.DataFrame({'day': ['Wed'], 'sales': [200]})

# Incremental append
updated_state = pd.concat([historical_state, delta_batch], ignore_index=True)
print("Incrementally Updated State:\n", updated_state)


Incrementally Updated State:
    day  sales
0  Mon    100
1  Tue    150
2  Wed    200


---
## Summary Table

| I/O & Storage Technique | Core Mechanism | Performance Advantage | Best Suited For |
| :--- | :--- | :--- | :--- |
| **Chunking Large Files** | Row-wise iterative block streaming | Prevents Out-Of-Memory (OOM) crashes | Multi-GB CSVs or JSONs exceeding RAM limits |
| **Columnar Formats (Parquet/Feather)** | Column-oriented storage & compression | Up to 80% smaller file sizes & rapid reads | Analytical data lakes and persistent storage |
| **Compressed Data Files** | On-the-fly codec compression/decompression | Reduced network transfer & storage footprints | Storing and transferring archived flat files |
| **Incremental Processing** | Micro-batching and delta partition appends | Minimal compute overhead vs full reloads | Real-time ETL pipelines and daily logs |